# **0: Menyiapkan Tempat Penyimpanan (Google Drive)**

In [1]:
# =====================================================================
# SESI 0: MOUNT GOOGLE DRIVE (PENYIMPANAN AMAN)
# =====================================================================
from google.colab import drive
import os

# 1. Hubungkan Colab ke Google Drive
print("Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

# 2. Tentukan folder khusus agar rapi di Drive Anda
folder_output = "/content/drive/MyDrive/Tugas_RAG_GapMatrix"
os.makedirs(folder_output, exist_ok=True)

print(f"\n✅ Berhasil! Semua hasil akan disimpan di folder: {folder_output}")

Menghubungkan ke Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Berhasil! Semua hasil akan disimpan di folder: /content/drive/MyDrive/Tugas_RAG_GapMatrix


# **1. Instalasi Tools, Library, dan Dependensi Sistem**

In [2]:
# =====================================================================
# SESI 1: INSTALASI DEPENDENSI SISTEM & LIBRARY PYTHON
# =====================================================================

# 1. Update paket Linux Colab dan instal software Tesseract OCR + Bahasa pendukung
!sudo apt-get update -qq
!sudo apt-get install -y tesseract-ocr tesseract-ocr-ind tesseract-ocr-eng -qq

# 2. Instal seluruh library Python untuk pemrosesan dokumen, vektor database, dan embedding
!pip install -q pymupdf pytesseract pillow chromadb sentence-transformers openai


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


# **2. Fungsi Ekstraksi Dokumen PDF Secara Hybrid**

In [3]:
# =====================================================================
# SESI 2: FUNGSI EKSTRAKSI PDF HYBRID + EKSTRAKSI DOI & JUDUL (V1.1)
# =====================================================================
import io
import os
import re
import pymupdf
import pytesseract
from PIL import Image

def baca_pdf_hybrid(path_file):
    if not os.path.exists(path_file):
        return f"Eror: File '{path_file}' tidak ditemukan."

    dokumen = pymupdf.open(path_file)
    teks_lengkap = ""

    print(f"Memulai pemrosesan dokumen: {path_file}")
    print(f"Total halaman: {len(dokumen)}")

    for indeks_hal, halaman in enumerate(dokumen):
        nomor_halaman = indeks_hal + 1
        teks_halaman = halaman.get_text()

        if len(teks_halaman.strip()) < 10:
            print(f"  -> [Halaman {nomor_halaman}] Teks digital kosong. Menjalankan Tesseract OCR...")
            pix = halaman.get_pixmap(dpi=150)
            img_data = pix.tobytes("png")
            gambar = Image.open(io.BytesIO(img_data))
            teks_ocr = pytesseract.image_to_string(gambar, lang='ind+eng')
            teks_lengkap += teks_ocr + "\n"
        else:
            print(f"  -> [Halaman {nomor_halaman}] Berhasil mengekstrak teks asli.")
            teks_lengkap += teks_halaman + "\n"

    return teks_lengkap

# --- FUNGSI BARU V1.1: Ekstrak DOI ---
def ekstrak_doi_langsung(path_file):
    try:
        doc = pymupdf.open(path_file)
        teks_halaman_1 = doc[0].get_text()
        match = re.search(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', teks_halaman_1, re.IGNORECASE)
        if match:
            return match.group(0)
        return "Tidak disebutkan"
    except Exception:
        return "Error ekstraksi DOI"

# --- FUNGSI BARU V1.1: Ekstrak Judul dari Halaman 1 ---
def ekstrak_judul_langsung(path_file):
    try:
        doc = pymupdf.open(path_file)
        teks_halaman_1 = doc[0].get_text()
        # Ambil baris pertama yang panjangnya lebih dari 20 karakter (asumsi judul paper)
        baris = teks_halaman_1.split('\n')
        for b in baris:
            b = b.strip()
            if len(b) > 20 and not b.lower().startswith(('http', 'doi', 'received', 'volume', 'page')):
                return b
        return "Tidak disebutkan"
    except Exception:
        return "Error ekstraksi Judul"

# **3. Fungsi Pemotongan Teks Berbasis Kata**

In [4]:
# =====================================================================
# SESI 3: FUNGSI CHUNKING (CHUNK KECIL AGAR AMAN TOKEN)
# =====================================================================
def chunking_akademik(teks, chunk_size=500, overlap=100): # <-- Ubah ke 500/100
    import re
    paragraf = re.split(r'\n\s*\n', teks)
    chunks = []
    current = ""
    for p in paragraf:
        if len(current) + len(p) < chunk_size:
            current += p + "\n\n"
        else:
            if current:
                chunks.append(current.strip())
            current = p + "\n\n"
    if current:
        chunks.append(current.strip())
    return chunks

# **4. Inisialisasi Vektor Database & Penyimpanan Dokumen Berkelanjutan**

In [9]:
# =====================================================================
# SESI 4: INISIALISASI CHROMADB & INGESTI (MANUAL EMBEDDING)
# =====================================================================
import chromadb
from sentence_transformers import SentenceTransformer
import os

path_database = "./chroma_db_rag"
client = chromadb.PersistentClient(path=path_database)

print("Memuat model embedding...")
model_embedding = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

koleksi = client.get_or_create_collection(name="koleksi_rag_pdf")

nama_file_sekarang = "jurnal02.pdf" # <-- UBAH SETIAP GANTI JURNAL

print(f"--- JALUR INGESTI DATA UNTUK: {nama_file_sekarang} ---")

if not os.path.exists(nama_file_sekarang):
    print(f"⚠️ Eror: Silakan upload file '{nama_file_sekarang}' ke Colab.")
else:
    teks_mentah = baca_pdf_hybrid(nama_file_sekarang)

    # Menggunakan chunk_size=700
    daftar_chunk = chunking_akademik(teks_mentah, chunk_size=700, overlap=150)
    print(f"✓ Berhasil memotong dokumen menjadi {len(daftar_chunk)} chunk.")

    ids_chunk = []
    metadata_chunk = []
    for indeks, chunk in enumerate(daftar_chunk):
        ids_chunk.append(f"{nama_file_sekarang}_chunk_{indeks}")
        metadata_chunk.append({"sumber": nama_file_sekarang})

    print("-> Sedang memproses vektor embedding...")
    embeddings = model_embedding.encode(daftar_chunk).tolist()

    print("-> Menyimpan ke database...")
    koleksi.add(
        documents=daftar_chunk,
        embeddings=embeddings,
        ids=ids_chunk,
        metadatas=metadata_chunk
    )

    print(f"🎉 SUKSES! File '{nama_file_sekarang}' telah diindeks.")
    print(f"Total akumulasi data di database saat ini: {koleksi.count()} chunk.")

Memuat model embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- JALUR INGESTI DATA UNTUK: jurnal02.pdf ---
Memulai pemrosesan dokumen: jurnal02.pdf
Total halaman: 60
  -> [Halaman 1] Berhasil mengekstrak teks asli.
  -> [Halaman 2] Berhasil mengekstrak teks asli.
  -> [Halaman 3] Berhasil mengekstrak teks asli.
  -> [Halaman 4] Berhasil mengekstrak teks asli.
  -> [Halaman 5] Berhasil mengekstrak teks asli.
  -> [Halaman 6] Berhasil mengekstrak teks asli.
  -> [Halaman 7] Berhasil mengekstrak teks asli.
  -> [Halaman 8] Berhasil mengekstrak teks asli.
  -> [Halaman 9] Berhasil mengekstrak teks asli.
  -> [Halaman 10] Berhasil mengekstrak teks asli.
  -> [Halaman 11] Berhasil mengekstrak teks asli.
  -> [Halaman 12] Berhasil mengekstrak teks asli.
  -> [Halaman 13] Berhasil mengekstrak teks asli.
  -> [Halaman 14] Berhasil mengekstrak teks asli.
  -> [Halaman 15] Berhasil mengekstrak teks asli.
  -> [Halaman 16] Berhasil mengekstrak teks asli.
  -> [Halaman 17] Berhasil mengekstrak teks asli.
  -> [Halaman 18] Berhasil mengekstrak teks asli.
  -

# **5. Fungsi Pencarian Informasi Relevan**

In [10]:
# =====================================================================
# SESI 5: RETRIEVAL (k=5)
# =====================================================================
def cari_chunk_relevan(pertanyaan, nama_sumber, k=5): # <-- k=5
    embedding_pertanyaan = model_embedding.encode([pertanyaan]).tolist()
    hasil = koleksi.query(
        query_embeddings=embedding_pertanyaan,
        n_results=k,
        where={"sumber": nama_sumber}
    )
    return hasil["documents"][0], hasil["metadatas"][0]

pertanyaan_pengguna = "Tolong ekstrak informasi Problem, Method, Dataset, Finding, Limitation, dan Future Work dari paper ini."

chunks, metadatas = cari_chunk_relevan(pertanyaan_pengguna, nama_file_sekarang, k=5)

print(f"Hasil pencarian data untuk: '{pertanyaan_pengguna}'\n")
print("================= DATA CHUNK RELEVAN =================")
for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
    sumber_file = meta.get("sumber", "Tidak diketahui")
    print(f"📌 [Rank {i+1}] File Asal: {sumber_file}")
    print("-" * 50)
    print(f"{chunk[:300]}...")
    print("-" * 50 + "\n")

Hasil pencarian data untuk: 'Tolong ekstrak informasi Problem, Method, Dataset, Finding, Limitation, dan Future Work dari paper ini.'

================= DATA CHUNK RELEVAN =================
📌 [Rank 1] File Asal: jurnal02.pdf
--------------------------------------------------
1 3
191 
Page 10 of 60
W. Chen et al.
Complementary components include loss aware calibration to match task loss (Nahshan 
et al. 2020), bit split with stitching for tighter reconstruction (Wang et al. 2020), random­
ized dropping to stabilize ultra low bit settings (Wei et al. 2023a), and outlier s...
--------------------------------------------------

📌 [Rank 2] File Asal: jurnal02.pdf
--------------------------------------------------
1 3
191 
Page 42 of 60
W. Chen et al.
both the base model and adapters, while the high memory usage of the pruned 4B model 
suggests potential overhead from managing sparse structures in the framework.
5.2.3  Synthesis and method-level takeaways
Synthesizing the findings from both 

# **6. Integrasi LLM Universal dan Pengujian Metrik Performa**

In [11]:
# =====================================================================
# SESI 6: GENERASI JAWABAN (PROMPT CERDAS V1.1)
# =====================================================================
import time
import json
from openai import OpenAI

def jawab_dengan_rag_universal(pertanyaan, client_llm, nama_model, nama_sumber):
    chunk_relevan, _ = cari_chunk_relevan(pertanyaan, nama_sumber)
    konteks = "\n\n".join(chunk_relevan)

    doi_terekstrak = ekstrak_doi_langsung(nama_sumber)
    judul_terekstrak = ekstrak_judul_langsung(nama_sumber)

    # PROMPT BARU: Memberikan panduan jika paper adalah "Survey" dan mencari sinonim "Limitation"
    prompt = f"""Anda adalah asisten peneliti. Ekstrak informasi dari Konteks di bawah.
Tuliskan jawaban secara utuh dan jelas.

PANDUAN KHUSUS:
1. Jika ini adalah paper "Survey/Review" dan tidak ada dataset eksperimen, tulis: "Survei literatur (tidak ada dataset eksperimen)".
2. Untuk mencari "Limitation", perhatikan juga bagian yang berjudul "Open Issues", "Challenges", atau "Tantangan".
3. Jika informasi benar-benar tidak ada di teks, barulah tulis "Tidak disebutkan".

Judul: {judul_terekstrak}
DOI: {doi_terekstrak}

Konteks:
{konteks}

Keluarkan hasil HANYA dalam format JSON valid:
{{
  "judul": "{judul_terekstrak}",
  "doi": "{doi_terekstrak}",
  "problem": "Masalah utama (jelaskan dengan kalimat lengkap)",
  "method": "Metode (jelaskan dengan kalimat lengkap)",
  "dataset": "Dataset (atau penjelasan jika survei)",
  "finding": "Temuan utama (jelaskan dengan kalimat lengkap)",
  "limitation": "Keterbatasan (atau 'Tidak disebutkan')",
  "future_work": "Saran (atau 'Tidak disebutkan')"
}}
Jawaban:"""

    print("=================== JAWABAN LLM (V1.1) ===================")
    waktu_mulai_request = time.perf_counter()

    response_stream = client_llm.chat.completions.create(
        model=nama_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},
        temperature=0.1,
        max_tokens=2000
    )

    waktu_token_pertama = None
    jawaban_akhir = ""
    token_output_count = 0
    token_input_total = 0
    token_keseluruhan = 0

    for chunk in response_stream:
        if hasattr(chunk, 'usage') and chunk.usage is not None:
            token_input_total = chunk.usage.prompt_tokens
            token_keseluruhan = chunk.usage.total_tokens
            continue

        if chunk.choices and chunk.choices[0].delta.content:
            token_teks = chunk.choices[0].delta.content
            if waktu_token_pertama is None:
                waktu_token_pertama = time.perf_counter()
            jawaban_akhir += token_teks
            token_output_count += 1
            print(token_teks, end="", flush=True)

    waktu_selesai_generasi = time.perf_counter()
    print("\n" + "="*51)

    ttft_ms = (waktu_token_pertama - waktu_mulai_request) * 1000 if waktu_token_pertama else 0
    waktu_decode_detik = waktu_selesai_generasi - waktu_token_pertama if waktu_token_pertama else 0
    tps = token_output_count / waktu_decode_detik if waktu_decode_detik > 0 else 0

    if token_input_total == 0:
        token_input_total = len(prompt) // 4
        token_keseluruhan = token_input_total + token_output_count

    print("\n📊 STATISTIK KINERJA & BIAYA TOKEN:")
    print(f"⏱️ Time to First Token (TTFT) : {ttft_ms:.2f} ms")
    print(f"⚡ Throughput (TPS)           : {tps:.2f} tokens/sec")
    print(f"📥 Token Input (Prompt)       : {token_input_total} token")
    print(f"📤 Token Output (Completion)  : {token_output_count} token")
    print(f"🔄 Total Token Terpakai       : {token_keseluruhan} token")

    biaya_estimasi = ((token_input_total * 0.15) + (token_output_count * 0.60)) / 1_000_000
    print(f"💰 Estimasi Biaya Sesi Ini    : ${biaya_estimasi:.7f} USD")

    metrik_performa = {
        "ttft_ms": ttft_ms, "tps": tps, "token_input": token_input_total,
        "token_output": token_output_count, "total_token": token_keseluruhan,
        "biaya_usd": biaya_estimasi
    }

    return jawaban_akhir, konteks, metrik_performa

# =====================================================================
# EKSEKUSI PENGUJIAN
# =====================================================================
api_key_kamu = "MASUKAN API KEY DISINI"
client_pilihan = OpenAI(api_key=api_key_kamu, base_url="https://api.groq.com/openai/v1")
model_pilihan = "openai/gpt-oss-120b"

if 'pertanyaan_pengguna' in locals():
    jawaban, konteks_terpilih, metrik_terakhir = jawab_dengan_rag_universal(
        pertanyaan=pertanyaan_pengguna,
        client_llm=client_pilihan,
        nama_model=model_pilihan,
        nama_sumber=nama_file_sekarang
    )
else:
    print("⚠️ Eror: Variabel 'pertanyaan_pengguna' tidak ditemukan.")

=================== JAWABAN LLM (V1.1) ===================
{
  "judul": "Artificial Intelligence Review (2026) 59:191",
  "doi": "10.1007/s10462-026-11538-1",
  "problem": "Paper ini membahas tantangan utama dalam mengompresi model bahasa besar (LLM) untuk inferensi pada perangkat dengan sumber daya terbatas, khususnya masalah akurasi, latensi, dan jejak memori ketika menerapkan kuantisasi ultra‑low‑bit (di bawah 4‑bit) serta kesulitan menjaga kinerja yang stabil di berbagai platform (GPU, CPU, dan akselerator edge).",
  "method": "Sebagai tinjauan literatur, artikel ini mengklasifikasikan dan menganalisis metode‑metode kuantisasi LLM, meliputi Post‑Training Quantization (PTQ), Quantization‑Aware Training (QAT), representasi hibrida, serta teknik penanganan outlier (mis. SqueezeLLM, SpQR, SINQ). Penulis menyajikan tabel perbandingan, timeline (Fig. 3), dan panduan praktis untuk memilih strategi kompresi berdasarkan batasan perangkat, serta menekankan pentingnya melaporkan latensi, memo

# **7. Logging Pengujian & Ekspor Data (Excel/CSV) untuk Bab Analisis Dokumen**

In [12]:
# =====================================================================
# SESI 7: AUTOMATED LOGGING & EKSPOR DATA KE GOOGLE DRIVE
# =====================================================================
import pandas as pd
import datetime
import os
import json

def simpan_log_pengujian(metrik_data, folder_tujuan):
    variabel_wajib = ['pertanyaan_pengguna', 'jawaban', 'konteks_terpilih', 'model_pilihan']
    for var in variabel_wajib:
        if var not in globals():
            print(f"⚠️ Eror: Variabel '{var}' tidak ditemukan. Pastikan Sesi 5 dan Sesi 6 sudah dijalankan.")
            return

    # Agar nama file baru tidak menumpuk dengan data percobaan lama
    path_file_excel = os.path.join(folder_tujuan, "log_pengujian_rag_v2.xlsx")
    path_file_csv = os.path.join(folder_tujuan, "log_pengujian_rag_v2.csv")

    waktu_sekarang = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Parsing JSON
    try:
        json_str = jawaban.strip().removeprefix('```json').removesuffix('```').strip()
        parsed_jawaban = json.loads(json_str)
    except Exception as e:
        print(f"⚠️ Peringatan: Gagal parsing JSON. Error: {e}")
        parsed_jawaban = {
            "judul": "Gagal Parse", "doi": "Gagal Parse",
            "problem": "Gagal Parse", "method": "Gagal Parse", "dataset": "Gagal Parse",
            "finding": jawaban, "limitation": "Gagal Parse", "future_work": "Gagal Parse"
        }

    # Menyusun struktur baris data baru (PENAMBAHAN JUDUL & DOI)
    data_baru = {
        "Waktu Pengujian": [waktu_sekarang],
        "Model LLM": [model_pilihan],
        "Nama File": [nama_file_sekarang],
        "Judul": [parsed_jawaban.get("judul", "")],
        "DOI": [parsed_jawaban.get("doi", "")],
        "Problem": [parsed_jawaban.get("problem", "")],
        "Method": [parsed_jawaban.get("method", "")],
        "Dataset": [parsed_jawaban.get("dataset", "")],
        "Temuan": [parsed_jawaban.get("finding", "")],
        "Keterbatasan": [parsed_jawaban.get("limitation", "")],
        "Future Work": [parsed_jawaban.get("future_work", "")],
        "Konteks RAG": [konteks_terpilih],
        "TTFT (ms)": [round(metrik_data.get('ttft_ms', 0), 2)],
        "Throughput (TPS)": [round(metrik_data.get('tps', 0), 2)],
        "Token Input": [metrik_data.get('token_input', 0)],
        "Token Output": [metrik_data.get('token_output', 0)],
        "Total Token": [metrik_data.get('total_token', 0)],
        "Estimasi Biaya (USD)": [metrik_data.get('biaya_usd', 0)]
    }

    df_baru = pd.DataFrame(data_baru)

    # Simpan (kumulatif)
    if os.path.exists(path_file_excel):
        df_lama = pd.read_excel(path_file_excel)
        df_total = pd.concat([df_lama, df_baru], ignore_index=True)
    else:
        df_total = df_baru

    if os.path.exists(path_file_csv):
        df_baru.to_csv(path_file_csv, mode='a', header=False, index=False)
    else:
        df_baru.to_csv(path_file_csv, mode='w', header=True, index=False)

    df_total.to_excel(path_file_excel, index=False)

    print("=================== LOGGING SYSTEM ===================")
    print(f"🎉 SUKSES! Data Jurnal {nama_file_sekarang} berhasil dicatat.")
    print(f"📁 File Excel tersimpan di: {path_file_excel}")
    print(f"📊 Total baris saat ini: {len(df_total)} data.")
    print("======================================================")

    print("\nPreview Data Gap Matrix (3 Baris Terakhir):")
    # Menampilkan kolom penting termasuk DOI
    display(df_total[["Nama File", "DOI", "Problem", "Method"]].tail(3))

# --- EKSEKUSI PENCATATAN LOG ---
simpan_log_pengujian(metrik_data=metrik_terakhir, folder_tujuan=folder_output)

=================== LOGGING SYSTEM ===================
🎉 SUKSES! Data Jurnal jurnal02.pdf berhasil dicatat.
📁 File Excel tersimpan di: /content/drive/MyDrive/Tugas_RAG_GapMatrix/log_pengujian_rag_v2.xlsx
📊 Total baris saat ini: 12 data.

Preview Data Gap Matrix (3 Baris Terakhir):


,Nama File,DOI,Problem,Method
9,jurnal10.pdf,10.26811/didaktika.v9i3.1997,Menilai efektivitas perlakuan pembelajaran ter...,Desain eksperimental dengan dua kelompok belaj...
10,jurnal01.pdf,10.1109/OJCOMS.2024.3456549,LLM‑based Edge Intelligence (EI) menghadapi ta...,Paper ini merupakan tinjauan literatur yang se...
11,jurnal02.pdf,10.1007/s10462-026-11538-1,Paper ini membahas tantangan utama dalam mengo...,"Sebagai tinjauan literatur, artikel ini mengkl..."
